# Pricing & Discount Optimizer

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Generar price scenarios** using `GENERATOR()`
2. **Model price elasticity** with `POWER()` function
3. **Calculate optimal pricing** through SQL scenario analysis
4. **Test discount strategies** (100 combinations)
5. **Maximize revenue** with what-if analysis

---

## What You'll Build

A **pricing optimization system** that finds revenue-maximizing prices:
- 100 price/discount scenario generation
- Price elasticity curve modeling
- Revenue maximization analysis
- What-if comparisons vs. baseline
- No complex optimization algorithms - pure SQL!

---

## Technical Requirements

- **Snowflake account** with standard SQL access
- **Approximately 12 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features Used

- `GENERATOR(ROWCOUNT => 100)` - Create scenarios
- `POWER()` - Model price elasticity curves
- `UNIFORM()` - Random discount generation
- `GREATEST()` and `LEAST()` - Boundary constraints
- Scenario ranking - Find optimal combinations

Let's begin!

---

## Paso 1: Configuración del Entorno

### Price Optimization Challenge

Find optimal price/discount that maximizes revenue = price × volume

In [ ]:
CREATE DATABASE IF NOT EXISTS PRICING_OPTIMIZER_DB;
USE SCHEMA PRICING_OPTIMIZER_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL';
USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() as db;

---

## Paso 2: Create Tables

In [ ]:
CREATE OR REPLACE TABLE historical_pricing (
    price_date DATE,
    product_name VARCHAR(100),
    list_price DECIMAL(10,2),
    discount_pct FLOAT,
    net_price DECIMAL(10,2),
    units_sold INTEGER,
    revenue DECIMAL(15,2)
);

CREATE OR REPLACE TABLE price_elasticity_estimates (
    product_name VARCHAR(100) PRIMARY KEY,
    elasticity_coefficient FLOAT,
    baseline_price DECIMAL(10,2),
    baseline_volume INTEGER
);

---

## Paso 3: Generar Historical Pricing Data

### Qué Vamos a Crear

- **Elasticity estimates** for each product (how volume responds to price changes)
- **12 months** of historical pricing and volume data
- **Price-volume relationship** modeling using `POWER()` function

### Understanding Price Elasticity with `POWER()`

**Price elasticity** measures how demand changes when price changes. We use Snowflake's `POWER()` function to model this relationship:

```sql
new_volume = baseline_volume * POWER(
    (new_price / baseline_price),
    elasticity_coefficient
)
```

### Why `POWER()` for Elasticity?

The `POWER(base, exponent)` function calculates **base^exponent**, which perfectly models economic elasticity:

- **Elasticity = -1.2** means: 10% price decrease → 12% volume increase
- **Negative coefficients** = inverse relationship (lower price = higher volume)
- **Formula**: If price drops to 90% of baseline: `POWER(0.90, -1.2)` = 1.126 (12.6% volume increase)

### Real-World Example

**Xarelto** with elasticity **-1.2**:
- Baseline: 100,000 units/month at $950
- 10% discount → $855
- New volume: `100,000 * POWER(855/950, -1.2)` = **112,640 units**
- Result: 10% price cut drives 12.6% more prescriptions

### Technical Details

- **Xarelto**: -1.2 elasticity (moderately price-sensitive)
- **Eylea**: -1.5 elasticity (more price-sensitive)
- **Stivarga**: -1.8 elasticity (highly price-sensitive)
- Values from market research and historical data analysis

In [ ]:
-- Insert elasticity estimates
INSERT INTO price_elasticity_estimates VALUES
    ('Xarelto', -1.2, 950.00, 100000),
    ('Eylea', -1.5, 600.00, 80000),
    ('Stivarga', -1.8, 500.00, 50000);

-- Generar 12 months of pricing/volume data
INSERT INTO historical_pricing
WITH months AS (
    SELECT DATEADD('month', -SEQ4(), CURRENT_DATE()) as price_date
    FROM TABLE(GENERATOR(ROWCOUNT => 12))
),
products AS (SELECT product_name, elasticity_coefficient, baseline_price, baseline_volume
             FROM price_elasticity_estimates)
SELECT 
    m.price_date,
    p.product_name,
    p.baseline_price as list_price,
    UNIFORM(0, 25, RANDOM()) as discount_pct,
    ROUND(list_price * (1 - discount_pct/100), 2) as net_price,
    -- Apply elasticity: volume changes based on price
    FLOOR(p.baseline_volume * POWER(
        (net_price / p.baseline_price),
        p.elasticity_coefficient
    ) * (1 + UNIFORM(-0.1, 0.1, RANDOM()))) as units_sold,
    ROUND(net_price * units_sold, 2) as revenue
FROM months m CROSS JOIN products p;

SELECT product_name, COUNT(*) as months, 
       ROUND(AVG(revenue),0) as avg_monthly_revenue
FROM historical_pricing GROUP BY product_name;

---

## Paso 4: Generar Pricing Scenarios

### Qué Vamos a Hacer

Creating **100 pricing scenarios per product** to find the optimal price point that maximizes profit.

### How It Works

1. **`GENERATOR(ROWCOUNT => 100)`**: Creates 100 different discount scenarios (0-30%)
2. **Calculate net price**: Apply discount to list price
3. **`POWER()` predicts volume**: Model how volume changes at each price point
4. **Calculate revenue & profit**: Determine financial impact of each scenario

### The `POWER()` Function in Action

For each scenario, we use `POWER()` to predict demand at the new price:

```sql
predicted_volume = baseline_volume * POWER(
    (net_price / baseline_price),
    elasticity_coefficient
)
```

**Example Scenario for Xarelto**:
- Scenario #42: 15% discount
- Net price: $950 × 0.85 = **$807.50**
- Price ratio: 807.50 / 950 = **0.85**
- Volume calculation: `100,000 * POWER(0.85, -1.2)` = **119,200 units** (+19.2%)
- Revenue: $807.50 × 119,200 = **$96.3M** vs. $95M baseline (+1.4%)

### Why This Matters

By testing 100 scenarios, we can:
- **Find the sweet spot** where profit is maximized (not always highest price!)
- **Quantify trade-offs** between volume and margin
- **Make data-driven decisions** instead of guessing
- **Compare strategies** across different products with different elasticities

### Technical Note

The `POWER()` function handles the non-linear relationship between price and volume. A simple linear model would incorrectly assume 10% price cut = 10% volume increase. Real markets don't work that way!

In [ ]:
-- Generar 100 scenarios per product
CREATE OR REPLACE TABLE pricing_scenarios AS
WITH scenario_grid AS (
    SELECT 
        ROW_NUMBER() OVER (ORDER BY SEQ4()) as scenario_id,
        UNIFORM(0, 30, RANDOM()) as discount_pct
    FROM TABLE(GENERATOR(ROWCOUNT => 100))
),
scenario_calcs AS (
    SELECT 
        sg.scenario_id,
        pe.product_name,
        pe.baseline_price as list_price,
        sg.discount_pct,
        ROUND(list_price * (1 - discount_pct/100), 2) as net_price,
        -- Calculate expected volume using elasticity
        FLOOR(pe.baseline_volume * POWER(
            (net_price / pe.baseline_price),
            pe.elasticity_coefficient
        )) as predicted_volume,
        ROUND(net_price * predicted_volume, 2) as predicted_revenue,
        ROUND(predicted_revenue * 0.65, 2) as predicted_profit  -- 65% margin
    FROM scenario_grid sg
    CROSS JOIN price_elasticity_estimates pe
)
SELECT * FROM scenario_calcs
ORDER BY product_name, scenario_id;

-- Find optimal scenarios per product
SELECT product_name, discount_pct, net_price, 
       predicted_volume, predicted_revenue, predicted_profit
FROM pricing_scenarios
QUALIFY ROW_NUMBER() OVER (PARTITION BY product_name ORDER BY predicted_profit DESC) = 1;

---

## Paso 5: Revenue Impact Analysis

Compare scenarios against baseline

In [ ]:
-- Compare all scenarios to baseline
CREATE OR REPLACE VIEW scenario_analysis AS
WITH baseline AS (
    SELECT product_name, 
           baseline_price * baseline_volume as baseline_revenue,
           baseline_price * baseline_volume * 0.65 as baseline_profit
    FROM price_elasticity_estimates
)
SELECT 
    ps.product_name,
    ps.scenario_id,
    ps.discount_pct,
    ps.net_price,
    ps.predicted_revenue,
    ps.predicted_profit,
    b.baseline_profit,
    ROUND(ps.predicted_profit - b.baseline_profit, 0) as profit_vs_baseline,
    ROUND((ps.predicted_profit - b.baseline_profit) / b.baseline_profit * 100, 1) as profit_lift_pct,
    CASE 
        WHEN profit_lift_pct >= 10 THEN '🟢 Strong Opportunity'
        WHEN profit_lift_pct >= 5 THEN '🟡 Moderate Gain'
        WHEN profit_lift_pct >= 0 THEN '⚪ Slight Gain'
        ELSE '🔴 Worse Than Baseline'
    END as recommendation
FROM pricing_scenarios ps
JOIN baseline b ON ps.product_name = b.product_name;

-- Top 5 scenarios per product
SELECT product_name, discount_pct, profit_lift_pct, recommendation
FROM scenario_analysis
WHERE recommendation IN ('🟢 Strong Opportunity', '🟡 Moderate Gain')
QUALIFY ROW_NUMBER() OVER (PARTITION BY product_name ORDER BY profit_lift_pct DESC) <= 5;

---

## Paso 6: Interactive Pricing Dashboard

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("💰 Pricing & Discount Optimizer")

# Product selection
product = st.selectbox("Select Product", ['Xarelto', 'Eylea', 'Stivarga'])

# Optimal scenario
optimal = session.sql(f"""
    SELECT discount_pct, net_price, predicted_revenue, predicted_profit, profit_lift_pct
    FROM scenario_analysis
    WHERE product_name='{product}'
    ORDER BY predicted_profit DESC LIMIT 1
""").collect()[0]

col1, col2, col3 = st.columns(3)
with col1:
    st.metric("Optimal Discount", f"{optimal['DISCOUNT_PCT']:.1f}%")
with col2:
    st.metric("Optimal Net Price", f"${optimal['NET_PRICE']:.2f}")
with col3:
    st.metric("Profit Lift", f"+{optimal['PROFIT_LIFT_PCT']:.1f}%")

# Scenario comparison
st.subheader("📊 All Scenarios")
scenarios = session.sql(f"""
    SELECT discount_pct, predicted_profit, profit_lift_pct, recommendation
    FROM scenario_analysis
    WHERE product_name='{product}'
    ORDER BY predicted_profit DESC LIMIT 20
""").to_pandas()

st.dataframe(scenarios, use_container_width=True, hide_index=True)

# Price elasticity curve
st.subheader("📈 Price vs Revenue Curve")
curve = session.sql(f"""
    SELECT discount_pct, predicted_revenue
    FROM pricing_scenarios
    WHERE product_name='{product}'
    ORDER BY discount_pct
""").to_pandas()

st.line_chart(curve.set_index('DISCOUNT_PCT')['PREDICTED_REVENUE'])
st.caption("Shows revenue impact across discount levels")

---

##  Complete!

### What You Learned

1.  Model price elasticity with SQL
2.  Generar scenarios using `GENERATOR()`
3.  Calculate revenue impact across 100 scenarios
4.  Use `POWER()` function for elasticity curves
5.  Build what-if pricing dashboards

### Key SQL Functions

**Scenario Generation:**
```sql
SELECT UNIFORM(0, 30, RANDOM()) as discount_pct
FROM TABLE(GENERATOR(ROWCOUNT => 100))
```

**Elasticity Formula:**
```sql
predicted_volume = baseline_volume * POWER(
    (new_price / baseline_price),
    elasticity_coefficient
)
```

**Example**: elasticity = -1.2 means 10% price decrease → 12% volume increase

### Business Applications

- Promotional planning
- Contract negotiations with payers
- Volume discount strategies
- Competitive pricing response

[Snowflake GENERATOR Docs](https://docs.snowflake.com/en/sql-reference/functions/generator)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `PRICING_OPTIMIZER_DB` database (and all tables within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS PRICING_OPTIMIZER_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
